# Homework 5.1 - Steps 4-5: Data Splits and Nested Anchor Sets

This notebook performs Tasks 4 and 5 together because both tasks define the information that is available to the later decipherment procedure.

The goals are:

1. split the glyph-token corpus into training, validation, and test portions,
2. make the split contiguous and reproducible,
3. guarantee that later context windows are constructed entirely inside one partition,
4. construct nested random anchor sets at approximately 5%, 10%, 20%, and 40% of the distinct glyph types,
5. expose only those anchor translations to later fitting code.

The complete hidden map is read only to construct the permitted anchor files. It is not copied into the training, validation, or test split files.

## Split convention

Let the full glyph-token sequence be

$$
y_1,y_2,\ldots,y_L.
$$

We use a fixed contiguous split:

- 70% training,
- 15% validation,
- 15% test.

The partitions are contiguous rather than randomly shuffling token positions because the decoder later uses local context.

For a context radius $$m$$, a valid center position inside a partition must satisfy

$$
m \le i < N-m,
$$

where $$N$$ is the length of that partition.

The later dataset builder constructs windows separately inside each partition. Therefore no context window can cross a train/validation/test boundary.

## Anchor convention

The hidden inverse map is

$$
\Phi^{-1}:G\rightarrow T.
$$

A fixed seeded random permutation of the distinct glyph types is generated once.

The 5%, 10%, 20%, and 40% anchor sets are prefixes of that same permutation. Therefore they are nested:

$$
A_{5}\subseteq A_{10}\subseteq A_{20}\subseteq A_{40}.
$$

Percentages are computed over distinct glyph types, not glyph occurrences.

In [1]:
from pathlib import Path
import csv
import math

import numpy as np

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd
DATA_DIR = PROJECT_ROOT / "data"

DECIPHER_DIR = DATA_DIR / "decipherment_inputs"
EVALUATION_DIR = DATA_DIR / "evaluation_only"

GLYPH_SEQUENCE_PATH = DECIPHER_DIR / "meditations_glyph_token_sequence.csv"
GLYPH_VOCAB_PATH = DECIPHER_DIR / "glyph_vocabulary.csv"
HIDDEN_MAP_PATH = EVALUATION_DIR / "hidden_syllable_glyph_bijection.csv"

SPLIT_DIR = DECIPHER_DIR / "splits"
ANCHOR_DIR = DECIPHER_DIR / "anchors"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
ANCHOR_DIR.mkdir(parents=True, exist_ok=True)

required = [
    GLYPH_SEQUENCE_PATH,
    GLYPH_VOCAB_PATH,
    HIDDEN_MAP_PATH,
]

for path in required:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}\n"
            "Run the corrected Steps 1-3 before this notebook."
        )

print("Split output folder:", SPLIT_DIR.resolve())
print("Anchor output folder:", ANCHOR_DIR.resolve())

Split output folder: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\splits
Anchor output folder: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\anchors


## Load the glyph vocabulary and complete glyph-token sequence

In [2]:
glyph_vocab = []

with GLYPH_VOCAB_PATH.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        glyph_vocab.append({
            "glyph_id": int(row["glyph_id"]),
            "glyph_label": row["glyph_label"],
        })

glyph_vocab.sort(key=lambda r: r["glyph_id"])
K = len(glyph_vocab)

assert [r["glyph_id"] for r in glyph_vocab] == list(range(K))

full_sequence = []

with GLYPH_SEQUENCE_PATH.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for expected_position, row in enumerate(reader):
        position = int(row["position"])
        glyph_id = int(row["glyph_id"])
        glyph_label = row["glyph_label"]

        assert position == expected_position
        assert glyph_vocab[glyph_id]["glyph_label"] == glyph_label

        full_sequence.append({
            "original_position": position,
            "glyph_id": glyph_id,
            "glyph_label": glyph_label,
        })

L = len(full_sequence)

print("Distinct glyph types K:", f"{K:,}")
print("Total glyph-token occurrences L:", f"{L:,}")

Distinct glyph types K: 3,052
Total glyph-token occurrences L: 77,260


## Create the contiguous 70/15/15 split

In [3]:
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

assert abs(
    TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION - 1.0
) < 1e-12

train_end = int(round(TRAIN_FRACTION * L))
validation_end = int(round((TRAIN_FRACTION + VALIDATION_FRACTION) * L))

# Protect against rounding anomalies.
train_end = max(1, min(train_end, L - 2))
validation_end = max(train_end + 1, min(validation_end, L - 1))

partitions = {
    "train": full_sequence[:train_end],
    "validation": full_sequence[train_end:validation_end],
    "test": full_sequence[validation_end:],
}

assert sum(len(v) for v in partitions.values()) == L
assert partitions["train"][0]["original_position"] == 0
assert partitions["train"][-1]["original_position"] + 1 == partitions["validation"][0]["original_position"]
assert partitions["validation"][-1]["original_position"] + 1 == partitions["test"][0]["original_position"]

for name, rows in partitions.items():
    print(
        f"{name:<11}",
        f"start={rows[0]['original_position']:,}",
        f"end={rows[-1]['original_position']:,}",
        f"tokens={len(rows):,}",
        f"fraction={len(rows)/L:.4%}",
    )

train       start=0 end=54,081 tokens=54,082 fraction=70.0000%
validation  start=54,082 end=65,670 tokens=11,589 fraction=15.0000%
test        start=65,671 end=77,259 tokens=11,589 fraction=15.0000%


## Save the three partition token sequences

In [4]:
split_paths = {}

for name, rows in partitions.items():
    path = SPLIT_DIR / f"{name}_glyph_sequence.csv"
    split_paths[name] = path

    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "partition_position",
            "original_position",
            "glyph_id",
            "glyph_label",
        ])

        for partition_position, row in enumerate(rows):
            writer.writerow([
                partition_position,
                row["original_position"],
                row["glyph_id"],
                row["glyph_label"],
            ])

    print("Saved:", path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\splits\train_glyph_sequence.csv
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\splits\validation_glyph_sequence.csv
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\splits\test_glyph_sequence.csv


## Verify the context-boundary rule

The later decoder experiments use context radii

$$
m\in\{1,2,4\}.
$$

This cell reports how many valid center positions exist inside each partition.

No window is built from the full sequence after splitting. Each partition is handled independently.

In [5]:
CONTEXT_RADII_TO_VERIFY = [1, 2, 4]

window_summary = []

for m in CONTEXT_RADII_TO_VERIFY:
    width = 2 * m + 1

    for name, rows in partitions.items():
        n = len(rows)
        valid_centers = max(0, n - 2 * m)

        window_summary.append({
            "partition": name,
            "context_radius_m": m,
            "window_width": width,
            "partition_tokens": n,
            "valid_centers": valid_centers,
        })

        print(
            f"{name:<11} m={m:<2} "
            f"window={width:<2} "
            f"valid centers={valid_centers:,}"
        )

assert all(row["valid_centers"] > 0 for row in window_summary)

print()
print("Boundary verification passed: windows will be generated separately per partition.")

train       m=1  window=3  valid centers=54,080
validation  m=1  window=3  valid centers=11,587
test        m=1  window=3  valid centers=11,587
train       m=2  window=5  valid centers=54,078
validation  m=2  window=5  valid centers=11,585
test        m=2  window=5  valid centers=11,585
train       m=4  window=9  valid centers=54,074
validation  m=4  window=9  valid centers=11,581
test        m=4  window=9  valid centers=11,581

Boundary verification passed: windows will be generated separately per partition.


## Load the hidden map only for permitted anchor construction

In [6]:
hidden_rows = []

with HIDDEN_MAP_PATH.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        hidden_rows.append({
            "source_id": int(row["source_id"]),
            "syllable": row["syllable"],
            "glyph_id": int(row["glyph_id"]),
            "glyph_label": row["glyph_label"],
        })

hidden_rows.sort(key=lambda r: r["glyph_id"])

assert len(hidden_rows) == K
assert [r["glyph_id"] for r in hidden_rows] == list(range(K))
assert len({r["source_id"] for r in hidden_rows}) == K
assert len({r["syllable"] for r in hidden_rows}) == K

hidden_by_glyph = {
    row["glyph_id"]: row
    for row in hidden_rows
}

print("Loaded complete map for anchor construction only.")
print("No complete-map file will be written into decipherment_inputs/.")

Loaded complete map for anchor construction only.
No complete-map file will be written into decipherment_inputs/.


## Construct the nested random anchor sets

In [7]:
ANCHOR_RANDOM_SEED = 63845
ANCHOR_PERCENTAGES = [5, 10, 20, 40]

rng = np.random.default_rng(ANCHOR_RANDOM_SEED)
glyph_permutation = rng.permutation(K).tolist()

anchor_sets = {}
anchor_summary = []

previous = set()

for percentage in ANCHOR_PERCENTAGES:
    requested_fraction = percentage / 100.0
    count = max(1, int(round(requested_fraction * K)))

    # Prefix of one shared permutation guarantees nesting.
    glyph_ids = set(glyph_permutation[:count])

    assert previous.issubset(glyph_ids)
    previous = glyph_ids

    actual_percentage = 100.0 * len(glyph_ids) / K

    anchor_sets[percentage] = glyph_ids
    anchor_summary.append({
        "requested_percentage": percentage,
        "anchor_count": len(glyph_ids),
        "actual_percentage": actual_percentage,
        "vocabulary_size_K": K,
    })

    print(
        f"{percentage:>2}% requested -> "
        f"{len(glyph_ids):>5,} glyph types -> "
        f"{actual_percentage:.4f}% actual"
    )

print()
print("Nestedness verified.")

 5% requested ->   153 glyph types -> 5.0131% actual
10% requested ->   305 glyph types -> 9.9934% actual
20% requested ->   610 glyph types -> 19.9869% actual
40% requested -> 1,221 glyph types -> 40.0066% actual

Nestedness verified.


## Save only the revealed translations

Each anchor file contains exactly the revealed direct translations for that experiment.

The complete hidden map remains only in `evaluation_only/`.

In [8]:
anchor_paths = {}

for percentage in ANCHOR_PERCENTAGES:
    glyph_ids = anchor_sets[percentage]
    path = ANCHOR_DIR / f"anchors_{percentage:02d}.csv"
    anchor_paths[percentage] = path

    rows = [
        hidden_by_glyph[glyph_id]
        for glyph_id in sorted(glyph_ids)
    ]

    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "glyph_id",
            "glyph_label",
            "source_id",
            "syllable",
        ])

        for row in rows:
            writer.writerow([
                row["glyph_id"],
                row["glyph_label"],
                row["source_id"],
                row["syllable"],
            ])

    print("Saved:", path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\anchors\anchors_05.csv
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\anchors\anchors_10.csv
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\anchors\anchors_20.csv
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\anchors\anchors_40.csv


## Verify the saved anchor files and information restriction

In [9]:
loaded_anchor_sets = {}

for percentage, path in anchor_paths.items():
    rows = []

    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append({
                "glyph_id": int(row["glyph_id"]),
                "source_id": int(row["source_id"]),
                "syllable": row["syllable"],
            })

    glyph_ids = {r["glyph_id"] for r in rows}
    loaded_anchor_sets[percentage] = glyph_ids

    assert glyph_ids == anchor_sets[percentage]

    for row in rows:
        truth = hidden_by_glyph[row["glyph_id"]]
        assert row["source_id"] == truth["source_id"]
        assert row["syllable"] == truth["syllable"]

for small, large in zip(ANCHOR_PERCENTAGES[:-1], ANCHOR_PERCENTAGES[1:]):
    assert loaded_anchor_sets[small].issubset(loaded_anchor_sets[large])

# Verify no split file contains plaintext labels.
for path in split_paths.values():
    header = path.read_text(encoding="utf-8").splitlines()[0].lower()
    assert "syllable" not in header
    assert "source_id" not in header

print("Anchor files match the permitted revealed translations.")
print("Nestedness of saved files verified.")
print("Train/validation/test split files contain no plaintext syllable labels.")

Anchor files match the permitted revealed translations.
Nestedness of saved files verified.
Train/validation/test split files contain no plaintext syllable labels.


## Save split and anchor metadata

In [10]:
SPLIT_METADATA_PATH = DATA_DIR / "task4_split_metadata.csv"

with SPLIT_METADATA_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "partition",
        "start_original_position",
        "end_original_position",
        "token_count",
        "fraction_of_full_sequence",
    ])

    for name, rows in partitions.items():
        writer.writerow([
            name,
            rows[0]["original_position"],
            rows[-1]["original_position"],
            len(rows),
            len(rows) / L,
        ])


WINDOW_METADATA_PATH = DATA_DIR / "task4_context_window_verification.csv"

with WINDOW_METADATA_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "partition",
            "context_radius_m",
            "window_width",
            "partition_tokens",
            "valid_centers",
        ],
    )
    writer.writeheader()
    writer.writerows(window_summary)


ANCHOR_SUMMARY_PATH = DATA_DIR / "task5_anchor_summary.csv"

with ANCHOR_SUMMARY_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "requested_percentage",
            "anchor_count",
            "actual_percentage",
            "vocabulary_size_K",
        ],
    )
    writer.writeheader()
    writer.writerows(anchor_summary)

print("Saved:", SPLIT_METADATA_PATH.resolve())
print("Saved:", WINDOW_METADATA_PATH.resolve())
print("Saved:", ANCHOR_SUMMARY_PATH.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\task4_split_metadata.csv
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\task4_context_window_verification.csv
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\task5_anchor_summary.csv


## Steps 4-5 outputs

### Split files

`data/decipherment_inputs/splits/`

- `train_glyph_sequence.csv`
- `validation_glyph_sequence.csv`
- `test_glyph_sequence.csv`

These contain only glyph information.

### Allowed anchor files

`data/decipherment_inputs/anchors/`

- `anchors_05.csv`
- `anchors_10.csv`
- `anchors_20.csv`
- `anchors_40.csv`

These are the only direct glyph-to-syllable translations that later fitting code may read.

### Metadata

`data/`

- `task4_split_metadata.csv`
- `task4_context_window_verification.csv`
- `task5_anchor_summary.csv`

The full hidden map remains isolated in `data/evaluation_only/`.